# Insurance Claims Risk, Data Quality & Process Analytics

## Business Problem

An insurance company wants to understand why claim settlement delays are occurring, identify potential high-risk claim patterns, improve claims data quality, and identify process improvements across the claims lifecycle.

## Business Objectives

1. Assess the quality and reliability of claims data.
2. Identify drivers of claim settlement delays.
3. Analyze claim rejection patterns.
4. Identify unusual or high-risk claim patterns requiring further investigation.
5. Analyze repeat-claim behavior.
6. Identify operational bottlenecks in the claims process.
7. Recommend data-quality controls and process improvements.

## Key Stakeholders

- Claims Operations Manager
- Risk & Fraud Investigation Team
- Insurance Business Analyst
- Finance Team
- Data Management / Data Governance Team
- Senior Management

## 1. Environment Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Insurance Claims Data

In [2]:
file_path = "/content/insurance_claims_case_study.csv"

claims = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Rows:", claims.shape[0])
print("Columns:", claims.shape[1])

Dataset loaded successfully.
Rows: 5000
Columns: 18


In [3]:
claims.head()

,claim_id,policy_id,customer_id,policy_type,region,provider_id,customer_age,policy_start_date,incident_date,claim_date,settlement_date,incident_type,claim_status,rejection_reason,premium_amount,claim_amount,approved_amount,previous_claims
0,CLM100000,POL10357,CUS20692,Motor,East,PRV-23,62,2024-05-23,2025-01-30,2025-02-09,NaN,Glass Damage,Rejected,Suspected Misrepresentation,"6,877.61","53,087.09",0.00,0
1,CLM100001,POL13095,CUS20118,Health,South,PRV-18,23,2024-08-03,2024-12-18,2024-12-26,2025-02-15,Diagnostic,Settled,NaN,"7,435.47","124,374.94","89,674.60",0
2,CLM100002,POL12618,CUS22195,Home,West,PRV-28,59,2025-06-17,2026-01-11,2026-01-13,2026-03-21,Water Damage,Settled,NaN,"8,233.99","63,916.32","57,532.13",0
3,CLM100003,POL11755,CUS22390,Motor,South,PRV-29,40,2024-03-04,2024-12-17,2024-12-28,NaN,Collision,Rejected,Coverage Limit,800.00,"12,586.63",0.00,0
4,CLM100004,POL11732,CUS22704,Motor,North,NaN,48,2024-03-08,2024-03-23,2024-04-03,NaN,Collision,Rejected,Policy Exclusion,"4,654.34","34,065.27",0.00,1


## 3. Initial Data Audit

Before performing business analysis, the dataset was profiled to understand its structure, completeness, data types, duplicate records, and potential data-quality issues.

In [4]:
print("Dataset Shape:", claims.shape)

print("\nColumn Names:")
print(claims.columns.tolist())

print("\nData Types:")
print(claims.dtypes)

Dataset Shape: (5000, 18)

Column Names:
['claim_id', 'policy_id', 'customer_id', 'policy_type', 'region', 'provider_id', 'customer_age', 'policy_start_date', 'incident_date', 'claim_date', 'settlement_date', 'incident_type', 'claim_status', 'rejection_reason', 'premium_amount', 'claim_amount', 'approved_amount', 'previous_claims']

Data Types:
claim_id              object
policy_id             object
customer_id           object
policy_type           object
region                object
provider_id           object
customer_age           int64
policy_start_date     object
incident_date         object
claim_date            object
settlement_date       object
incident_type         object
claim_status          object
rejection_reason      object
premium_amount       float64
claim_amount         float64
approved_amount      float64
previous_claims        int64
dtype: object


In [5]:
missing_values = claims.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (missing_values / len(claims) * 100).round(2)
})

missing_summary[missing_summary["Missing_Count"] > 0]

,Missing_Count,Missing_Percentage
policy_id,35,0.70
customer_id,20,0.40
provider_id,25,0.50
settlement_date,2326,46.52
rejection_reason,4389,87.78


In [6]:
duplicate_claim_ids = claims["claim_id"].duplicated().sum()

print("Duplicate Claim IDs:", duplicate_claim_ids)

Duplicate Claim IDs: 25


## 4. Data Type Standardization

Date fields were converted from text/object format into standardized datetime fields to enable temporal validation, turnaround-time analysis, and claims lifecycle analysis.

In [7]:
date_columns = [
    "policy_start_date",
    "incident_date",
    "claim_date",
    "settlement_date"
]

for col in date_columns:
    claims[col] = pd.to_datetime(claims[col], errors="coerce")

claims[date_columns].dtypes

,0
policy_start_date,datetime64[ns]
incident_date,datetime64[ns]
claim_date,datetime64[ns]
settlement_date,datetime64[ns]


## 5. Business-Rule Validation of Missing Values

Not every missing value represents a data-quality issue. Missing fields were evaluated against the insurance claims lifecycle and relevant business rules.

In [8]:
claims.groupby("claim_status")["settlement_date"].agg(
    Total_Claims="size",
    Missing_Settlement_Date=lambda x: x.isna().sum()
)

,Total_Claims,Missing_Settlement_Date
claim_status,,
Approved,1060,1060
Pending,625,625
Rejected,641,641
Settled,2674,0


In [9]:
claims.groupby("claim_status")["rejection_reason"].agg(
    Total_Claims="size",
    Missing_Rejection_Reason=lambda x: x.isna().sum()
)

,Total_Claims,Missing_Rejection_Reason
claim_status,,
Approved,1060,1060
Pending,625,625
Rejected,641,30
Settled,2674,2674


In [10]:
settled_without_date = claims[
    (claims["claim_status"] == "Settled") &
    (claims["settlement_date"].isna())
]

rejected_without_reason = claims[
    (claims["claim_status"] == "Rejected") &
    (claims["rejection_reason"].isna())
]

print("Settled claims without settlement date:",
      len(settled_without_date))

print("Rejected claims without rejection reason:",
      len(rejected_without_reason))

Settled claims without settlement date: 0
Rejected claims without rejection reason: 30


## 6. Data Quality Rule Validation

Business rules were applied across key data-quality dimensions including completeness, uniqueness, validity, and consistency.

In [11]:
dq_checks = {
    "Duplicate Claim ID":
        claims["claim_id"].duplicated().sum(),

    "Missing Policy ID":
        claims["policy_id"].isna().sum(),

    "Missing Customer ID":
        claims["customer_id"].isna().sum(),

    "Missing Provider ID":
        claims["provider_id"].isna().sum(),

    "Invalid Claim Amount":
        (claims["claim_amount"] <= 0).sum(),

    "Invalid Premium Amount":
        (claims["premium_amount"] <= 0).sum(),

    "Approved Amount > Claim Amount":
        (claims["approved_amount"] > claims["claim_amount"]).sum(),

    "Settlement Before Claim Date":
        (claims["settlement_date"] < claims["claim_date"]).sum(),

    "Incident Before Policy Start":
        (claims["incident_date"] < claims["policy_start_date"]).sum(),

    "Claim Before Incident":
        (claims["claim_date"] < claims["incident_date"]).sum(),

    "Invalid Customer Age":
        ((claims["customer_age"] < 18) |
         (claims["customer_age"] > 100)).sum(),

    "Settled Claim Missing Settlement Date":
        len(settled_without_date),

    "Rejected Claim Missing Rejection Reason":
        len(rejected_without_reason)
}

dq_checks

{'Duplicate Claim ID': np.int64(25),
 'Missing Policy ID': np.int64(35),
 'Missing Customer ID': np.int64(20),
 'Missing Provider ID': np.int64(25),
 'Invalid Claim Amount': np.int64(6),
 'Invalid Premium Amount': np.int64(6),
 'Approved Amount > Claim Amount': np.int64(24),
 'Settlement Before Claim Date': np.int64(15),
 'Incident Before Policy Start': np.int64(0),
 'Claim Before Incident': np.int64(0),
 'Invalid Customer Age': np.int64(7),
 'Settled Claim Missing Settlement Date': 0,
 'Rejected Claim Missing Rejection Reason': 30}

## 7. Data Quality Assessment Report

The identified issues were classified by data-quality dimension, quantified by failure rate, and assigned a severity based on their potential business impact on claims processing, financial reporting, risk analysis, and customer records.

In [12]:
dq_report = pd.DataFrame([
    ["Duplicate Claim ID", "Uniqueness", 25, "Critical",
     "May double-count claims and distort claim cost reporting"],

    ["Missing Policy ID", "Completeness", 35, "Critical",
     "Claim cannot be reliably linked to an insurance policy"],

    ["Missing Customer ID", "Completeness", 20, "High",
     "Prevents reliable customer-level and repeat-claim analysis"],

    ["Missing Provider ID", "Completeness", 25, "Medium",
     "Limits provider-level performance and risk analysis"],

    ["Invalid Claim Amount", "Validity", 6, "Critical",
     "Can distort financial exposure and claims severity metrics"],

    ["Invalid Premium Amount", "Validity", 6, "High",
     "Affects premium-to-claim and risk calculations"],

    ["Approved Amount > Claim Amount", "Consistency", 24, "Critical",
     "May indicate payment, processing, or source-data errors"],

    ["Settlement Before Claim Date", "Consistency", 15, "Critical",
     "Violates the expected claims lifecycle sequence"],

    ["Invalid Customer Age", "Validity", 7, "High",
     "May affect customer segmentation and underwriting analysis"],

    ["Rejected Claim Missing Rejection Reason", "Conditional Completeness", 30, "High",
     "Reduces transparency of rejection analysis and operational reporting"]
], columns=[
    "DQ_Rule",
    "DQ_Dimension",
    "Failed_Records",
    "Severity",
    "Business_Impact"
])

dq_report["Failure_Rate_%"] = (
    dq_report["Failed_Records"] / len(claims) * 100
).round(2)

dq_report

,DQ_Rule,DQ_Dimension,Failed_Records,Severity,Business_Impact,Failure_Rate_%
0,Duplicate Claim ID,Uniqueness,25,Critical,May double-count claims and distort claim cost...,0.50
1,Missing Policy ID,Completeness,35,Critical,Claim cannot be reliably linked to an insuranc...,0.70
2,Missing Customer ID,Completeness,20,High,Prevents reliable customer-level and repeat-cl...,0.40
3,Missing Provider ID,Completeness,25,Medium,Limits provider-level performance and risk ana...,0.50
4,Invalid Claim Amount,Validity,6,Critical,Can distort financial exposure and claims seve...,0.12
5,Invalid Premium Amount,Validity,6,High,Affects premium-to-claim and risk calculations,0.12
6,Approved Amount > Claim Amount,Consistency,24,Critical,"May indicate payment, processing, or source-da...",0.48
7,Settlement Before Claim Date,Consistency,15,Critical,Violates the expected claims lifecycle sequence,0.30
8,Invalid Customer Age,Validity,7,High,May affect customer segmentation and underwrit...,0.14
9,Rejected Claim Missing Rejection Reason,Conditional Completeness,30,High,Reduces transparency of rejection analysis and...,0.60


### Overall Data Quality Score

A rule-level pass rate was calculated to provide a simple measure of data quality across the validated business rules.

In [13]:
dq_report["Passed_Records"] = len(claims) - dq_report["Failed_Records"]

dq_report["Pass_Rate_%"] = (
    dq_report["Passed_Records"] / len(claims) * 100
).round(2)

overall_dq_score = dq_report["Pass_Rate_%"].mean()

print(f"Overall Data Quality Score: {overall_dq_score:.2f}%")

Overall Data Quality Score: 99.61%


In [14]:
dimension_summary = (
    dq_report
    .groupby("DQ_Dimension")
    .agg(
        Total_Failures=("Failed_Records", "sum"),
        Average_Pass_Rate=("Pass_Rate_%", "mean")
    )
    .round(2)
    .reset_index()
)

dimension_summary

,DQ_Dimension,Total_Failures,Average_Pass_Rate
0,Completeness,80,99.47
1,Conditional Completeness,30,99.40
2,Consistency,39,99.61
3,Uniqueness,25,99.50
4,Validity,19,99.87


## 8. Data Remediation

The raw dataset was preserved before remediation. Data-quality issues were corrected only where a defensible business rule was available; ambiguous records were flagged rather than silently imputed.

In [15]:
claims_raw = claims.copy()
claims_clean = claims.copy()

print("Raw dataset:", claims_raw.shape)
print("Working copy:", claims_clean.shape)

Raw dataset: (5000, 18)
Working copy: (5000, 18)


In [16]:
duplicate_records = claims_clean[
    claims_clean["claim_id"].duplicated(keep=False)
].sort_values("claim_id")

duplicate_records.head(10)

,claim_id,policy_id,customer_id,policy_type,region,provider_id,customer_age,policy_start_date,incident_date,claim_date,settlement_date,incident_type,claim_status,rejection_reason,premium_amount,claim_amount,approved_amount,previous_claims
574,CLM100574,POL10028,CUS22497,Motor,East,PRV-07,65,2024-04-07,2024-11-09,2024-11-18,2025-01-19,Glass Damage,Settled,NaN,"3,673.25","29,400.87","17,303.04",1
575,CLM100574,POL11494,CUS21887,Motor,Central,PRV-09,18,2024-04-06,2024-05-31,2024-06-09,2024-07-20,Glass Damage,Settled,NaN,"6,581.23","85,241.86","70,394.08",1
787,CLM100787,POL12744,CUS22136,Home,North,PRV-14,33,2024-12-09,2025-02-12,2025-02-19,2025-03-03,Fire,Settled,NaN,"22,440.82","140,791.93","116,056.02",0
788,CLM100787,POL12178,CUS21904,Home,South,PRV-20,41,2024-08-04,2025-03-24,2025-03-25,2025-04-14,Fire,Settled,NaN,"6,556.70","49,597.90","38,375.15",0
799,CLM100799,POL11769,CUS21470,Health,Central,PRV-25,41,2024-06-14,2024-10-21,2024-10-27,2024-12-06,Surgery,Settled,NaN,"2,459.73","30,174.31","24,145.43",0
800,CLM100799,POL10048,CUS20435,Health,North,PRV-13,50,2025-01-01,2025-09-03,2025-09-10,NaT,Hospitalization,Rejected,Late Notification,"8,201.08","62,265.37",0.00,1
948,CLM100948,POL13791,CUS23272,Motor,West,PRV-15,67,2024-05-01,2025-01-06,2025-01-16,NaT,Glass Damage,Rejected,Insufficient Documents,"5,287.04","86,895.91",0.00,1
949,CLM100948,POL11112,CUS21770,Health,West,PRV-15,21,2024-11-21,2025-08-24,2025-09-02,2025-10-22,Hospitalization,Settled,NaN,"1,427.71","29,016.82","27,947.03",1
1383,CLM101383,POL13587,CUS20093,Home,North,PRV-08,24,2024-04-09,2024-05-10,2024-05-11,NaT,Storm,Approved,NaN,"4,561.79","40,657.76","35,265.44",3
1384,CLM101383,POL11545,CUS20857,Motor,East,PRV-03,64,2025-02-26,2025-07-29,2025-07-31,NaT,Third Party,Approved,NaN,"7,245.36","153,042.69","104,462.77",0


In [17]:
print("Rows involved in duplicate Claim IDs:",
      len(duplicate_records))

print("Unique duplicated Claim IDs:",
      duplicate_records["claim_id"].nunique())

Rows involved in duplicate Claim IDs: 50
Unique duplicated Claim IDs: 25


## 9. Duplicate Investigation

Duplicate claim identifiers were investigated before remediation. Records sharing the same Claim ID were compared across other business attributes to distinguish exact duplicate records from identifier/key collisions.

In [18]:
exact_duplicates = claims_clean.duplicated().sum()

duplicate_id_rows = claims_clean[
    claims_clean["claim_id"].duplicated(keep=False)
]

print("Exact duplicate rows:", exact_duplicates)
print("Rows with duplicated Claim IDs:", len(duplicate_id_rows))
print("Duplicated Claim IDs:", duplicate_id_rows["claim_id"].nunique())

Exact duplicate rows: 0
Rows with duplicated Claim IDs: 50
Duplicated Claim IDs: 25


In [19]:
claims_clean["duplicate_claim_id_flag"] = (
    claims_clean["claim_id"].duplicated(keep=False)
)

claims_clean[
    claims_clean["duplicate_claim_id_flag"]
][["claim_id", "policy_id", "customer_id", "claim_status"]].head(10)

,claim_id,policy_id,customer_id,claim_status
574,CLM100574,POL10028,CUS22497,Settled
575,CLM100574,POL11494,CUS21887,Settled
787,CLM100787,POL12744,CUS22136,Settled
788,CLM100787,POL12178,CUS21904,Settled
799,CLM100799,POL11769,CUS21470,Settled
800,CLM100799,POL10048,CUS20435,Rejected
948,CLM100948,POL13791,CUS23272,Rejected
949,CLM100948,POL11112,CUS21770,Settled
1383,CLM101383,POL13587,CUS20093,Approved
1384,CLM101383,POL11545,CUS20857,Approved


In [20]:
claims_clean["duplicate_claim_id_flag"] = (
    claims_clean["claim_id"].duplicated(keep=False)
)

claims_clean[
    claims_clean["duplicate_claim_id_flag"]
][["claim_id", "policy_id", "customer_id", "claim_status"]].head(10)

,claim_id,policy_id,customer_id,claim_status
574,CLM100574,POL10028,CUS22497,Settled
575,CLM100574,POL11494,CUS21887,Settled
787,CLM100787,POL12744,CUS22136,Settled
788,CLM100787,POL12178,CUS21904,Settled
799,CLM100799,POL11769,CUS21470,Settled
800,CLM100799,POL10048,CUS20435,Rejected
948,CLM100948,POL13791,CUS23272,Rejected
949,CLM100948,POL11112,CUS21770,Settled
1383,CLM101383,POL13587,CUS20093,Approved
1384,CLM101383,POL11545,CUS20857,Approved


I did not automatically delete duplicate claim IDs because the associated records represented different policies and customers. I treated them as identifier-integrity exceptions requiring source-system investigation.

In [21]:
claims_clean["missing_policy_flag"] = claims_clean["policy_id"].isna()
claims_clean["missing_customer_flag"] = claims_clean["customer_id"].isna()
claims_clean["missing_provider_flag"] = claims_clean["provider_id"].isna()

claims_clean[
    [
        "missing_policy_flag",
        "missing_customer_flag",
        "missing_provider_flag"
    ]
].sum()

,0
missing_policy_flag,35
missing_customer_flag,20
missing_provider_flag,25


In [22]:
claims_clean["missing_policy_flag"] = claims_clean["policy_id"].isna()
claims_clean["missing_customer_flag"] = claims_clean["customer_id"].isna()
claims_clean["missing_provider_flag"] = claims_clean["provider_id"].isna()

claims_clean[
    [
        "missing_policy_flag",
        "missing_customer_flag",
        "missing_provider_flag"
    ]
].sum()

,0
missing_policy_flag,35
missing_customer_flag,20
missing_provider_flag,25


In [23]:
claims_clean["invalid_claim_amount_flag"] = (
    claims_clean["claim_amount"] <= 0
)

claims_clean["invalid_premium_flag"] = (
    claims_clean["premium_amount"] <= 0
)

claims_clean["approval_exceeds_claim_flag"] = (
    claims_clean["approved_amount"] > claims_clean["claim_amount"]
)

claims_clean[
    [
        "invalid_claim_amount_flag",
        "invalid_premium_flag",
        "approval_exceeds_claim_flag"
    ]
].sum()

,0
invalid_claim_amount_flag,6
invalid_premium_flag,6
approval_exceeds_claim_flag,24


In [24]:
claims_clean["settlement_before_claim_flag"] = (
    claims_clean["settlement_date"] < claims_clean["claim_date"]
)

claims_clean["incident_before_policy_flag"] = (
    claims_clean["incident_date"] < claims_clean["policy_start_date"]
)

claims_clean["claim_before_incident_flag"] = (
    claims_clean["claim_date"] < claims_clean["incident_date"]
)

claims_clean[
    [
        "settlement_before_claim_flag",
        "incident_before_policy_flag",
        "claim_before_incident_flag"
    ]
].sum()

,0
settlement_before_claim_flag,15
incident_before_policy_flag,0
claim_before_incident_flag,0


In [25]:
claims_clean["missing_rejection_reason_flag"] = (
    (claims_clean["claim_status"] == "Rejected") &
    (claims_clean["rejection_reason"].isna())
)

claims_clean["missing_settlement_date_flag"] = (
    (claims_clean["claim_status"] == "Settled") &
    (claims_clean["settlement_date"].isna())
)

claims_clean["invalid_age_flag"] = (
    (claims_clean["customer_age"] < 18) |
    (claims_clean["customer_age"] > 100)
)

In [26]:
dq_flag_columns = [
    "duplicate_claim_id_flag",
    "missing_policy_flag",
    "missing_customer_flag",
    "missing_provider_flag",
    "invalid_claim_amount_flag",
    "invalid_premium_flag",
    "approval_exceeds_claim_flag",
    "settlement_before_claim_flag",
    "incident_before_policy_flag",
    "claim_before_incident_flag",
    "missing_rejection_reason_flag",
    "missing_settlement_date_flag",
    "invalid_age_flag"
]

claims_clean["dq_issue_count"] = (
    claims_clean[dq_flag_columns].sum(axis=1)
)

claims_clean["dq_exception_flag"] = (
    claims_clean["dq_issue_count"] > 0
)

print(
    "Records with at least one DQ issue:",
    claims_clean["dq_exception_flag"].sum()
)

print(
    "DQ Exception Rate:",
    round(claims_clean["dq_exception_flag"].mean() * 100, 2),
    "%"
)

Records with at least one DQ issue: 208
DQ Exception Rate: 4.16 %


## 11. Analytical Dataset Preparation

A separate analytical dataset was created by excluding records containing critical financial, identifier, or lifecycle inconsistencies that could distort downstream business analysis. The original and exception datasets were preserved for auditability.

In [27]:
critical_flags = [
    "duplicate_claim_id_flag",
    "missing_policy_flag",
    "missing_customer_flag",
    "invalid_claim_amount_flag",
    "invalid_premium_flag",
    "approval_exceeds_claim_flag",
    "settlement_before_claim_flag"
]

claims_analysis = claims_clean[
    ~claims_clean[critical_flags].any(axis=1)
].copy()

print("Original records:", len(claims_clean))
print("Analytical records:", len(claims_analysis))
print(
    "Excluded from analytical dataset:",
    len(claims_clean) - len(claims_analysis)
)

Original records: 5000
Analytical records: 4854
Excluded from analytical dataset: 146


## 12. Feature Engineering

Business-relevant analytical features were derived to evaluate settlement performance, financial severity, repeat-claim behavior, and potential risk indicators.

In [28]:
claims_analysis["settlement_days"] = (
    claims_analysis["settlement_date"] -
    claims_analysis["claim_date"]
).dt.days

claims_analysis["days_to_report"] = (
    claims_analysis["claim_date"] -
    claims_analysis["incident_date"]
).dt.days

claims_analysis["policy_age_at_claim"] = (
    claims_analysis["claim_date"] -
    claims_analysis["policy_start_date"]
).dt.days

claims_analysis[
    [
        "claim_id",
        "settlement_days",
        "days_to_report",
        "policy_age_at_claim"
    ]
].head()

,claim_id,settlement_days,days_to_report,policy_age_at_claim
0,CLM100000,NaN,10,262
1,CLM100001,51.00,8,145
2,CLM100002,67.00,2,210
3,CLM100003,NaN,11,299
4,CLM100004,NaN,11,26


In [29]:
claims_analysis["claim_to_premium_ratio"] = (
    claims_analysis["claim_amount"] /
    claims_analysis["premium_amount"]
)

claims_analysis["approval_ratio"] = np.where(
    claims_analysis["claim_amount"] > 0,
    claims_analysis["approved_amount"] /
    claims_analysis["claim_amount"],
    np.nan
)

claims_analysis[
    [
        "claim_amount",
        "premium_amount",
        "claim_to_premium_ratio",
        "approval_ratio"
    ]
].describe()

,claim_amount,premium_amount,claim_to_premium_ratio,approval_ratio
count,"4,854.00","4,854.00","4,854.00","4,854.00"
mean,"53,109.89","5,534.72",11.67,0.58
std,"50,188.14","6,047.98",6.13,0.36
min,"1,621.12",800.00,2.03,0.00
25%,"23,064.76","1,971.14",7.05,0.00
50%,"39,534.24","3,761.69",9.47,0.70
75%,"66,193.90","6,968.93",14.41,0.85
max,"974,191.92","120,000.00",33.27,1.00


In [30]:
SLA_DAYS = 30

claims_analysis["delayed_claim_flag"] = (
    (claims_analysis["claim_status"] == "Settled") &
    (claims_analysis["settlement_days"] > SLA_DAYS)
)

claims_analysis["repeat_claim_flag"] = (
    claims_analysis["previous_claims"] >= 2
)

claims_analysis["early_policy_claim_flag"] = (
    claims_analysis["policy_age_at_claim"] <= 30
)

claims_analysis[
    [
        "delayed_claim_flag",
        "repeat_claim_flag",
        "early_policy_claim_flag"
    ]
].sum()

,0
delayed_claim_flag,1621
repeat_claim_flag,921
early_policy_claim_flag,427


## 13. Rule-Based Claim Risk Indicators

A transparent rule-based risk indicator was developed to prioritize claims for further review. The score is not intended to classify or confirm fraud; it identifies claims exhibiting combinations of unusual characteristics that may warrant additional investigation.

In [31]:
claims_analysis["high_claim_ratio_flag"] = (
    claims_analysis["claim_to_premium_ratio"] >= 10
)

claim_threshold = claims_analysis["claim_amount"].quantile(0.95)

claims_analysis["high_value_claim_flag"] = (
    claims_analysis["claim_amount"] >= claim_threshold
)

print("95th percentile claim threshold:",
      round(claim_threshold, 2))

95th percentile claim threshold: 139340.75


In [32]:
claims_analysis["risk_score"] = (
    claims_analysis["repeat_claim_flag"].astype(int) * 2 +
    claims_analysis["early_policy_claim_flag"].astype(int) * 2 +
    claims_analysis["high_claim_ratio_flag"].astype(int) * 2 +
    claims_analysis["high_value_claim_flag"].astype(int) * 1 +
    (claims_analysis["days_to_report"] > 7).astype(int) * 1
)

In [33]:
claims_analysis["risk_segment"] = pd.cut(
    claims_analysis["risk_score"],
    bins=[-1, 2, 4, 8],
    labels=["Low", "Medium", "High"]
)

claims_analysis["risk_segment"].value_counts()

,count
risk_segment,
Low,3340
Medium,1258
High,256


## 14. Claims Portfolio Analysis

In [34]:
portfolio_summary = pd.DataFrame({
    "Metric": [
        "Total Claims",
        "Total Claimed Amount",
        "Total Approved Amount",
        "Average Claim Amount",
        "Median Claim Amount",
        "Rejected Claims",
        "Settled Claims"
    ],

    "Value": [
        len(claims_analysis),
        claims_analysis["claim_amount"].sum(),
        claims_analysis["approved_amount"].sum(),
        claims_analysis["claim_amount"].mean(),
        claims_analysis["claim_amount"].median(),
        (claims_analysis["claim_status"] == "Rejected").sum(),
        (claims_analysis["claim_status"] == "Settled").sum()
    ]
})

portfolio_summary

,Metric,Value
0,Total Claims,"4,854.00"
1,Total Claimed Amount,"257,795,411.18"
2,Total Approved Amount,"146,589,245.21"
3,Average Claim Amount,"53,109.89"
4,Median Claim Amount,"39,534.24"
5,Rejected Claims,620.00
6,Settled Claims,"2,595.00"


In [35]:
policy_analysis = (
    claims_analysis
    .groupby("policy_type")
    .agg(
        Claims=("claim_id", "count"),
        Total_Claim_Amount=("claim_amount", "sum"),
        Average_Claim=("claim_amount", "mean"),
        Average_Previous_Claims=("previous_claims", "mean")
    )
    .round(2)
    .sort_values("Total_Claim_Amount", ascending=False)
)

policy_analysis

,Claims,Total_Claim_Amount,Average_Claim,Average_Previous_Claims
policy_type,,,,
Health,1686,"99,327,541.15","58,913.13",0.77
Motor,1858,"84,667,474.65","45,569.15",0.78
Home,833,"64,953,025.85","77,974.82",0.82
Travel,477,"8,847,369.53","18,547.94",0.81


In [36]:
settled_claims = claims_analysis[
    claims_analysis["claim_status"] == "Settled"
].copy()

delay_by_policy = (
    settled_claims
    .groupby("policy_type")
    .agg(
        Settled_Claims=("claim_id", "count"),
        Avg_Settlement_Days=("settlement_days", "mean"),
        Median_Settlement_Days=("settlement_days", "median"),
        Delayed_Claims=("delayed_claim_flag", "sum")
    )
    .round(2)
)

delay_by_policy["Delay_Rate_%"] = (
    delay_by_policy["Delayed_Claims"] /
    delay_by_policy["Settled_Claims"] * 100
).round(2)

delay_by_policy.sort_values(
    "Delay_Rate_%",
    ascending=False
)

,Settled_Claims,Avg_Settlement_Days,Median_Settlement_Days,Delayed_Claims,Delay_Rate_%
policy_type,,,,,
Travel,258,40.57,40.00,166,64.34
Motor,992,39.93,41.00,627,63.21
Home,451,39.30,39.00,282,62.53
Health,894,38.85,38.50,546,61.07


In [37]:
delay_rca = (
    settled_claims
    .groupby(["policy_type", "region"])
    .agg(
        Claims=("claim_id", "count"),
        Avg_Settlement_Days=("settlement_days", "mean"),
        Delayed_Claims=("delayed_claim_flag", "sum")
    )
    .reset_index()
)

delay_rca["Delay_Rate_%"] = (
    delay_rca["Delayed_Claims"] /
    delay_rca["Claims"] * 100
).round(2)

delay_rca.sort_values(
    ["Delay_Rate_%", "Claims"],
    ascending=[False, False]
).head(10)

,policy_type,region,Claims,Avg_Settlement_Days,Delayed_Claims,Delay_Rate_%
18,Travel,South,60,47.67,46,76.67
9,Home,West,94,43.14,67,71.28
8,Home,South,84,41.90,56,66.67
0,Health,Central,173,40.64,114,65.90
19,Travel,West,61,39.52,40,65.57
14,Motor,West,180,40.83,116,64.44
10,Motor,Central,198,40.11,127,64.14
13,Motor,South,203,38.90,129,63.55
17,Travel,North,43,38.84,27,62.79
11,Motor,East,204,41.24,128,62.75


In [48]:
provider_delay = (
    settled_claims
    .groupby("provider_id")
    .agg(
        Claims=("claim_id", "count"),
        Avg_Settlement_Days=("settlement_days", "mean"),
        Delayed_Claims=("delayed_claim_flag", "sum")
    )
    .reset_index()
)

provider_delay["Delay_Rate_%"] = (
    provider_delay["Delayed_Claims"] /
    provider_delay["Claims"] * 100
).round(2)

provider_delay[
    provider_delay["Claims"] >= 20
].sort_values(
    ["Delay_Rate_%", "Claims"],
    ascending=[False, False]
).head(10)

,provider_id,Claims,Avg_Settlement_Days,Delayed_Claims,Delay_Rate_%
11,PRV-12,79,41.37,56,70.89
13,PRV-14,99,40.34,68,68.69
17,PRV-18,76,41.82,52,68.42
12,PRV-13,98,40.71,66,67.35
16,PRV-17,87,39.82,58,66.67
9,PRV-10,100,41.81,66,66.00
22,PRV-23,88,41.24,58,65.91
3,PRV-04,83,41.53,54,65.06
4,PRV-05,83,42.02,54,65.06
27,PRV-28,94,39.06,61,64.89


## 15. Claim Rejection Analysis

In [39]:
rejected_claims = claims_analysis[
    claims_analysis["claim_status"] == "Rejected"
].copy()

rejection_summary = (
    rejected_claims["rejection_reason"]
    .fillna("Missing Reason")
    .value_counts()
    .reset_index()
)

rejection_summary.columns = [
    "Rejection_Reason",
    "Claims"
]

rejection_summary

,Rejection_Reason,Claims
0,Coverage Limit,131
1,Late Notification,120
2,Suspected Misrepresentation,119
3,Policy Exclusion,114
4,Insufficient Documents,106
5,Missing Reason,30


In [40]:
rejection_by_policy = pd.crosstab(
    rejected_claims["policy_type"],
    rejected_claims["rejection_reason"].fillna("Missing Reason")
)

rejection_by_policy

rejection_reason,Coverage Limit,Insufficient Documents,Late Notification,Missing Reason,Policy Exclusion,Suspected Misrepresentation
policy_type,,,,,,
Health,50,32,36,11,43,35
Home,27,19,22,6,24,21
Motor,42,45,48,10,36,46
Travel,12,10,14,3,11,17


## 16. Repeat Claim Analysis

In [41]:
repeat_comparison = (
    claims_analysis
    .groupby("repeat_claim_flag")
    .agg(
        Claims=("claim_id", "count"),
        Avg_Claim_Amount=("claim_amount", "mean"),
        Avg_Claim_to_Premium=("claim_to_premium_ratio", "mean"),
        Avg_Risk_Score=("risk_score", "mean")
    )
    .round(2)
)

repeat_comparison

,Claims,Avg_Claim_Amount,Avg_Claim_to_Premium,Avg_Risk_Score
repeat_claim_flag,,,,
False,3933,"53,357.96",11.69,1.51
True,921,"52,050.56",11.60,3.48


In [49]:
risk_summary = (
    claims_analysis
    .groupby("risk_segment", observed=True)
    .agg(
        Claims=("claim_id", "count"),
        Avg_Claim_Amount=("claim_amount", "mean"),
        Total_Claim_Amount=("claim_amount", "sum"),
        Avg_Previous_Claims=("previous_claims", "mean"),
        Avg_Risk_Score=("risk_score", "mean")
    )
    .round(2)
)

risk_summary

,Claims,Avg_Claim_Amount,Total_Claim_Amount,Avg_Previous_Claims,Avg_Risk_Score
risk_segment,,,,,
Low,3340,"49,520.85","165,399,629.67",0.60,1.09
Medium,1258,"60,746.73","76,419,388.40",1.06,3.33
High,256,"62,407.79","15,976,393.11",1.91,5.21


In [42]:
high_risk_claims = (
    claims_analysis[
        claims_analysis["risk_segment"] == "High"
    ][[
        "claim_id",
        "policy_id",
        "customer_id",
        "policy_type",
        "region",
        "provider_id",
        "claim_amount",
        "premium_amount",
        "previous_claims",
        "policy_age_at_claim",
        "days_to_report",
        "risk_score"
    ]]
    .sort_values(
        ["risk_score", "claim_amount"],
        ascending=False
    )
)

high_risk_claims.head(20)

,claim_id,policy_id,customer_id,policy_type,region,provider_id,claim_amount,premium_amount,previous_claims,policy_age_at_claim,days_to_report,risk_score
4397,CLM104397,POL11034,CUS21837,Home,North,PRV-11,"169,222.77","13,704.62",2,23,1,7
2211,CLM102211,POL13097,CUS20014,Home,East,PRV-01,"160,760.26","10,276.16",2,15,0,7
1639,CLM101639,POL11092,CUS21089,Health,West,PRV-19,"72,283.77","3,291.03",2,12,10,7
3541,CLM103541,POL12320,CUS23493,Motor,West,PRV-09,"54,740.82","2,097.91",2,26,11,7
1305,CLM101305,POL12686,CUS21238,Motor,Central,PRV-02,"49,005.85","3,485.01",2,27,11,7
1198,CLM101198,POL13211,CUS20555,Health,West,PRV-05,"38,577.06","1,233.88",2,29,8,7
3703,CLM103703,POL13623,CUS20239,Motor,West,PRV-23,"31,381.28","1,050.35",2,20,8,7
2672,CLM102672,POL13406,CUS22493,Travel,East,PRV-22,"30,228.89","1,486.97",2,20,11,7
1483,CLM101483,POL10448,CUS21970,Home,South,PRV-04,"26,323.19","2,520.86",2,25,11,7
150,CLM100150,POL13422,CUS20554,Health,South,PRV-23,"19,970.53","1,838.61",2,17,11,7


## 18. Claim Severity Outlier Analysis

In [43]:
Q1 = claims_analysis["claim_amount"].quantile(0.25)
Q3 = claims_analysis["claim_amount"].quantile(0.75)

IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

claims_analysis["claim_amount_outlier_flag"] = (
    claims_analysis["claim_amount"] > upper_bound
)

print("IQR upper bound:", round(upper_bound, 2))
print(
    "Claim amount outliers:",
    claims_analysis["claim_amount_outlier_flag"].sum()
)

IQR upper bound: 130887.6
Claim amount outliers: 284


In [44]:
claims_analysis[
    claims_analysis["claim_amount_outlier_flag"]
][[
    "claim_id",
    "policy_type",
    "region",
    "claim_amount",
    "risk_segment"
]].sort_values(
    "claim_amount",
    ascending=False
).head(15)

,claim_id,policy_type,region,claim_amount,risk_segment
1494,CLM101494,Home,West,"974,191.92",Medium
249,CLM100249,Home,East,"812,137.97",Low
1418,CLM101418,Home,North,"633,392.83",Low
532,CLM100532,Health,East,"546,589.05",Low
3374,CLM103374,Health,West,"457,231.04",Low
3393,CLM103393,Health,South,"437,220.43",Medium
2399,CLM102399,Home,East,"395,683.99",Low
3826,CLM103826,Home,East,"395,281.72",High
146,CLM100146,Home,West,"393,554.82",Medium
1458,CLM101458,Health,East,"376,224.73",Medium


## 19. Business Requirements

### BR-01
The solution should identify the major drivers of claim settlement delays across policy type, region, and provider.

### BR-02
The solution should identify claims exhibiting defined risk indicators for further investigation.

### BR-03
The solution should analyze claim rejection reasons across insurance products.

### BR-04
The solution should identify repeat-claim behavior and compare it with other claims.

### BR-05
The solution should monitor critical data-quality exceptions affecting claims reporting and analysis.

### BR-06
The solution should identify high-value and unusual claim patterns without automatically classifying them as fraudulent.

## 20. User Stories and Acceptance Criteria

### User Story 1
As a Claims Operations Manager, I want to identify claims exceeding the analytical settlement threshold so that delayed claims can be prioritized for review.

Acceptance Criteria:
- Settlement turnaround time must be calculated from claim date to settlement date.
- Settled claims exceeding 30 days must be flagged.
- Results must be segmentable by policy type, region, and provider.

### User Story 2
As a Risk Analyst, I want to identify claims exhibiting multiple risk indicators so that potentially unusual claims can be prioritized for further investigation.

Acceptance Criteria:
- Risk indicators must be transparent and rule-based.
- High-risk classification must not be treated as confirmed fraud.
- Analysts must be able to review the underlying indicators contributing to the score.

### User Story 3
As a Data Steward, I want critical data-quality exceptions to be identified so that unreliable records can be investigated before downstream reporting.

Acceptance Criteria:
- Duplicate claim identifiers must be flagged.
- Missing critical identifiers must be identified.
- Financial and lifecycle consistency rules must be validated.

## 20. User Stories and Acceptance Criteria

### User Story 1
As a Claims Operations Manager, I want to identify claims exceeding the analytical settlement threshold so that delayed claims can be prioritized for review.

Acceptance Criteria:
- Settlement turnaround time must be calculated from claim date to settlement date.
- Settled claims exceeding 30 days must be flagged.
- Results must be segmentable by policy type, region, and provider.

### User Story 2
As a Risk Analyst, I want to identify claims exhibiting multiple risk indicators so that potentially unusual claims can be prioritized for further investigation.

Acceptance Criteria:
- Risk indicators must be transparent and rule-based.
- High-risk classification must not be treated as confirmed fraud.
- Analysts must be able to review the underlying indicators contributing to the score.

### User Story 3
As a Data Steward, I want critical data-quality exceptions to be identified so that unreliable records can be investigated before downstream reporting.

Acceptance Criteria:
- Duplicate claim identifiers must be flagged.
- Missing critical identifiers must be identified.
- Financial and lifecycle consistency rules must be validated.

## 21. As-Is Claims Process

Claim Submitted
→ Manual Data Validation
→ Document Review
→ Claim Assessment
→ Approval / Rejection
→ Settlement
→ Reporting

### Identified Pain Points

- Data-quality exceptions may enter downstream reporting.
- Duplicate claim identifiers create reconciliation risk.
- Claims are not prioritized using transparent risk indicators.
- Delayed claims may not be escalated proactively.
- Missing rejection reasons reduce operational transparency.
- Provider-level delay patterns may remain hidden in aggregate reporting.

## 23. User Acceptance Testing

In [45]:
uat_cases = pd.DataFrame([
    ["UAT-01", "BR-05",
     "Duplicate Claim ID",
     "Duplicate Claim ID should be flagged",
     "Pass"],

    ["UAT-02", "BR-05",
     "Missing Policy ID",
     "Record should be identified as a DQ exception",
     "Pass"],

    ["UAT-03", "BR-05",
     "Approved amount exceeds claim amount",
     "Financial consistency exception should be raised",
     "Pass"],

    ["UAT-04", "BR-05",
     "Settlement date before claim date",
     "Lifecycle consistency exception should be raised",
     "Pass"],

    ["UAT-05", "BR-01",
     "Settled claim exceeds 30 days",
     "Delayed claim flag should equal True",
     "Pass"],

    ["UAT-06", "BR-03",
     "Rejected claim has no rejection reason",
     "Conditional completeness exception should be raised",
     "Pass"],

    ["UAT-07", "BR-02",
     "Claim meets multiple risk indicators",
     "Risk score and segment should be generated",
     "Pass"],

    ["UAT-08", "BR-04",
     "Previous claims >= 2",
     "Repeat claim flag should equal True",
     "Pass"]
], columns=[
    "Test_Case_ID",
    "Requirement_ID",
    "Scenario",
    "Expected_Result",
    "Test_Status"
])

uat_cases

,Test_Case_ID,Requirement_ID,Scenario,Expected_Result,Test_Status
0,UAT-01,BR-05,Duplicate Claim ID,Duplicate Claim ID should be flagged,Pass
1,UAT-02,BR-05,Missing Policy ID,Record should be identified as a DQ exception,Pass
2,UAT-03,BR-05,Approved amount exceeds claim amount,Financial consistency exception should be raised,Pass
3,UAT-04,BR-05,Settlement date before claim date,Lifecycle consistency exception should be raised,Pass
4,UAT-05,BR-01,Settled claim exceeds 30 days,Delayed claim flag should equal True,Pass
5,UAT-06,BR-03,Rejected claim has no rejection reason,Conditional completeness exception should be r...,Pass
6,UAT-07,BR-02,Claim meets multiple risk indicators,Risk score and segment should be generated,Pass
7,UAT-08,BR-04,Previous claims >= 2,Repeat claim flag should equal True,Pass


In [46]:
rtm = pd.DataFrame([
    ["BR-01", "Settlement Delay Analysis", "UAT-05", "Validated"],
    ["BR-02", "Risk Indicator Analysis", "UAT-07", "Validated"],
    ["BR-03", "Rejection Analysis", "UAT-06", "Validated"],
    ["BR-04", "Repeat Claim Analysis", "UAT-08", "Validated"],
    ["BR-05", "Data Quality Framework", "UAT-01 to UAT-04", "Validated"],
    ["BR-06", "Outlier & Risk Review", "Analytical Review", "Validated"]
], columns=[
    "Requirement_ID",
    "Analysis_Component",
    "Validation",
    "Status"
])

rtm

,Requirement_ID,Analysis_Component,Validation,Status
0,BR-01,Settlement Delay Analysis,UAT-05,Validated
1,BR-02,Risk Indicator Analysis,UAT-07,Validated
2,BR-03,Rejection Analysis,UAT-06,Validated
3,BR-04,Repeat Claim Analysis,UAT-08,Validated
4,BR-05,Data Quality Framework,UAT-01 to UAT-04,Validated
5,BR-06,Outlier & Risk Review,Analytical Review,Validated


## 24. Business Recommendations

1. Implement automated data-quality controls at claim intake for critical identifiers, financial values, and lifecycle dates.

2. Establish an exception-management workflow for records failing critical data-quality rules rather than silently correcting ambiguous records.

3. Introduce proactive settlement monitoring to identify claims approaching or exceeding operational turnaround thresholds.

4. Use transparent risk indicators to prioritize claims for additional review while avoiding automatic fraud classification.

5. Monitor settlement performance by policy type, region, and provider to identify operational bottlenecks hidden by portfolio-level averages.

6. Standardize rejection-reason capture for rejected claims to improve root-cause analysis and operational reporting.

7. Maintain auditable separation between raw source data, DQ exceptions, and analysis-ready datasets.

8. Assign ownership for critical claims data elements and periodically monitor data-quality performance.

In [50]:
dq_report.to_csv(
    "/content/data_quality_report.csv",
    index=False
)

claims_clean[
    claims_clean["dq_exception_flag"]
].to_csv(
    "/content/data_quality_exceptions.csv",
    index=False
)

claims_analysis.to_csv(
    "/content/claims_analysis_ready.csv",
    index=False
)

high_risk_claims.to_csv(
    "/content/high_risk_claims_for_review.csv",
    index=False
)

uat_cases.to_csv(
    "/content/uat_test_cases.csv",
    index=False
)

rtm.to_csv(
    "/content/requirements_traceability_matrix.csv",
    index=False
)

print("All project outputs saved successfully.")

All project outputs saved successfully.


## 25. Executive Summary

This project evaluated insurance claims data across three connected areas: data reliability, claims operations, and risk prioritization.

The analysis established business-aware data-quality controls, separated unreliable records from the analytical dataset, evaluated settlement and rejection patterns, and developed transparent rule-based risk indicators for further investigation.

The project also translated analytical findings into business requirements, user stories, acceptance criteria, process improvements, UAT scenarios, and a requirement traceability matrix.

The objective was not simply to build a dashboard, but to demonstrate how reliable data, business analysis, and structured analytical reasoning can support better claims operations and risk decisions.